# Raw Events from S3

This notebook reads XML event data from the `events-raw` S3 directory, flattens the nested XML structure (including Play elements at multiple nesting levels), and saves the results as Delta tables in `bundesliga-2022-2023.batch`.

In [0]:
# Read all XML files from the events-raw S3 directory
# rowTag="Event" gives one row per <Event> element, with all child types as nullable struct columns
s3_path = "s3://bundesliga-2022-2023-data/events-raw/"

event_df = (
    spark.read.format("xml")
    .option("rowTag", "Event")
    .option("encoding", "UTF-8")
    .load(s3_path)
)

print(f"Total events loaded: {event_df.count()}")
print(f"Total columns: {len(event_df.columns)}")
print(f"\nEvent attributes:")
for c in event_df.columns:
    if c.startswith("_"):
        print(f"  {c}")
print(f"\nEvent child types:")
for c in event_df.columns:
    if not c.startswith("_"):
        print(f"  {c}")

In [0]:
from pyspark.sql.functions import col, when, lit, to_json
from pyspark.sql.types import StructType, ArrayType

# ── 1. Flatten ALL event fields (one row per event, every child field extracted) ──
# Goal: ZERO information loss — every field from every event type in the raw XML
# is present as a column in the output table.

event_attrs = [
    col("_EventId").alias("event_id"),
    col("_MatchId").alias("match_id"),
    col("_EventTime").alias("event_time"),
    col("_X-Position").alias("x_position"),
    col("_Y-Position").alias("y_position"),
    col("_X-Source-Position").alias("x_source_position"),
    col("_Y-Source-Position").alias("y_source_position"),
    col("_X-PositionFromTracking").alias("x_position_from_tracking"),
    col("_Y-PositionFromTracking").alias("y_position_from_tracking"),
    col("_StartFrame").alias("start_frame"),
    col("_EndFrame").alias("end_frame"),
    col("_CalculatedFrame").alias("calculated_frame"),
    col("_CalculatedTimestamp").alias("calculated_timestamp"),
]

# All event types (direct children of <Event>)
event_types = [
    "BallClaiming", "BallDeflection", "Caution", "CautionTeamofficial",
    "ChanceWithoutShot", "CornerKick", "Delete", "FairPlay", "FinalWhistle",
    "Foul", "FreeKick", "GoalDisallowed", "GoalKick", "KickOff", "Nutmeg",
    "Offside", "OtherBallAction", "OtherPlayerAction", "Penalty",
    "PenaltyNotAwarded", "Play", "PlayerNotSentOff", "PossessionLossBeforeGoal",
    "RefereeBall", "Run", "ShotAtGoal", "SitterPrevented", "SpectacularPlay",
    "Substitution", "TacklingGame", "ThrowIn", "VideoAssistantAction",
]

# Build event_type column: first non-null child element
event_type_expr = lit(None).alias("_tmp")
for et in event_types:
    event_type_expr = when(col(et).isNotNull(), lit(et)).otherwise(event_type_expr)

# ── Helper: convert CamelCase / _attr to snake_case ──
def to_snake(name):
    name = name.lstrip("_")
    result = []
    for i, c in enumerate(name):
        if c.isupper() and i > 0:
            result.append("_")
        result.append(c.lower())
    return "".join(result)

# ── Dynamically flatten every field from every event type ──
# Strategy:
#   - Scalar fields  -> select directly, prefixed as eventtype__fieldname
#   - Nested structs -> flatten ONE level deeper, prefixed as eventtype__sub__field
#   - Arrays of struct -> convert to JSON string (preserves all data, no row explosion)
#   - Arrays of scalar  -> convert to JSON string
# This guarantees zero information loss from the raw XML.

flattened_cols = []
fields_summary = []  # (event_type, field_path, spark_type, strategy)

for et in event_types:
    field = event_df.schema[et]
    if not isinstance(field.dataType, StructType):
        continue

    prefix = to_snake(et)

    for sub_field in field.dataType.fields:
        col_name = to_snake(sub_field.name)
        sub_type = sub_field.dataType

        if isinstance(sub_type, StructType):
            # Nested struct -- flatten recursively up to 2 more levels
            for nested_field in sub_type.fields:
                nested_col_name = to_snake(nested_field.name)
                nested_type = nested_field.dataType

                if isinstance(nested_type, StructType):
                    # 3rd level: struct inside nested struct
                    for deep_field in nested_type.fields:
                        deep_col_name = to_snake(deep_field.name)
                        deep_type = deep_field.dataType
                        full_name = f"{prefix}__{col_name}__{nested_col_name}__{deep_col_name}"

                        if isinstance(deep_type, (ArrayType, StructType)):
                            flattened_cols.append(
                                to_json(col(f"{et}.{sub_field.name}.{nested_field.name}.{deep_field.name}")).alias(full_name)
                            )
                            fields_summary.append((et, f"{sub_field.name}.{nested_field.name}.{deep_field.name}", str(deep_type), "json"))
                        else:
                            flattened_cols.append(
                                col(f"{et}.{sub_field.name}.{nested_field.name}.{deep_field.name}").alias(full_name)
                            )
                            fields_summary.append((et, f"{sub_field.name}.{nested_field.name}.{deep_field.name}", str(deep_type), "scalar"))

                elif isinstance(nested_type, ArrayType):
                    full_name = f"{prefix}__{col_name}__{nested_col_name}"
                    flattened_cols.append(
                        to_json(col(f"{et}.{sub_field.name}.{nested_field.name}")).alias(full_name)
                    )
                    fields_summary.append((et, f"{sub_field.name}.{nested_field.name}", str(nested_type), "json"))
                else:
                    full_name = f"{prefix}__{col_name}__{nested_col_name}"
                    flattened_cols.append(
                        col(f"{et}.{sub_field.name}.{nested_field.name}").alias(full_name)
                    )
                    fields_summary.append((et, f"{sub_field.name}.{nested_field.name}", str(nested_type), "scalar"))

        elif isinstance(sub_type, ArrayType):
            # Array of struct or scalar -> JSON string to preserve all data
            full_name = f"{prefix}__{col_name}"
            flattened_cols.append(
                to_json(col(f"{et}.{sub_field.name}")).alias(full_name)
            )
            fields_summary.append((et, sub_field.name, str(sub_type), "json"))

        else:
            # Scalar field
            full_name = f"{prefix}__{col_name}"
            flattened_cols.append(
                col(f"{et}.{sub_field.name}").alias(full_name)
            )
            fields_summary.append((et, sub_field.name, str(sub_type), "scalar"))

# Build the comprehensive events DataFrame
all_cols = event_attrs + [event_type_expr.alias("event_type")] + flattened_cols
events_flat_df = event_df.select(*all_cols)

print(f"Total columns in enriched events table: {len(events_flat_df.columns)}")
print(f"  Event attributes: {len(event_attrs)}")
print(f"  Event type label: 1")
print(f"  Flattened detail fields: {len(flattened_cols)}")
print(f"    Scalar fields: {sum(1 for _, _, _, s in fields_summary if s == 'scalar')}")
print(f"    JSON-wrapped fields: {sum(1 for _, _, _, s in fields_summary if s == 'json')}")

print("\nEvents by type:")
events_flat_df.groupBy("event_type").count().orderBy(col("count").desc()).show(40, truncate=False)

In [0]:
# ── 2. Flatten plays (one row per Play element from all nesting levels) ──
# Play elements appear at 6 levels:
#   1. Direct child of Event:         Event.Play
#   2. Inside KickOff:                Event.KickOff.Play
#   3. Inside FreeKick:               Event.FreeKick.Play
#   4. Inside CornerKick:             Event.CornerKick.Play
#   5. Inside GoalKick:               Event.GoalKick.Play
#   6. Inside ThrowIn:                Event.ThrowIn.Play
# Each nesting level may have a slightly different Play schema, so we dynamically
# check which fields exist before selecting them.

from pyspark.sql.types import StructType

play_levels = [
    ("Play", "direct"),
    ("KickOff.Play", "KickOff"),
    ("FreeKick.Play", "FreeKick"),
    ("CornerKick.Play", "CornerKick"),
    ("GoalKick.Play", "GoalKick"),
    ("ThrowIn.Play", "ThrowIn"),
]

event_cols = [
    col("_EventId").alias("event_id"),
    col("_MatchId").alias("match_id"),
    col("_EventTime").alias("event_time"),
    col("_X-Position").alias("x_position"),
    col("_Y-Position").alias("y_position"),
    col("_X-Source-Position").alias("x_source_position"),
    col("_Y-Source-Position").alias("y_source_position"),
]

# All possible Play fields with their target aliases and types
play_scalar_attrs = [
    ("_Player", "player", "string"),
    ("_Recipient", "recipient", "string"),
    ("_Team", "team", "string"),
    ("_BallPossessionPhase", "ball_possession_phase", "long"),
    ("_Evaluation", "evaluation", "string"),
    ("_PlayAngle", "play_angle", "double"),
    ("_Height", "height", "string"),
    ("_Distance", "distance", "string"),
    ("_FromOpenPlay", "from_open_play", "boolean"),
    ("_SemiField", "semi_field", "boolean"),
    ("_PenaltyBox", "penalty_box", "boolean"),
    ("_FlatCross", "flat_cross", "boolean"),
    ("_PlayOrigin", "play_origin", "string"),
    ("_Rotation", "play_rotation", "string"),
    ("_GoalKeeperAction", "goalkeeper_action", "string"),
]

pass_attrs = [
    ("_FreeKickLayup", "pass_free_kick_layup", "boolean"),
    ("_Direction", "pass_direction", "string"),
    ("_OneTwo", "pass_one_two", "string"),
]

cross_attrs = [
    ("_Side", "cross_side", "string"),
    ("_GoalKeeper", "cross_goalkeeper", "string"),
    ("_GoalKeeperInterference", "cross_goalkeeper_interference", "string"),
]

def get_play_schema(struct_type, path_parts):
    """Navigate the DataFrame schema to find the struct type at the given path."""
    current = struct_type
    for part in path_parts:
        found = False
        for f in current.fields:
            if f.name == part:
                current = f.dataType
                found = True
                break
        if not found:
            return None
    return current

def build_play_df(event_df, play_path, context_label):
    """Build a DataFrame for Play elements at a specific nesting level."""
    path_parts = play_path.split(".")
    play_type = get_play_schema(event_df.schema, path_parts)
    if play_type is None or not isinstance(play_type, StructType):
        return None
    
    available_fields = set(f.name for f in play_type.fields)
    select_exprs = list(event_cols) + [lit(context_label).alias("play_context")]
    
    # Scalar Play attributes
    for attr, alias, dtype in play_scalar_attrs:
        if attr in available_fields:
            select_exprs.append(col(f"{play_path}.{attr}").alias(alias))
        else:
            select_exprs.append(lit(None).cast(dtype).alias(alias))
    
    # Pass sub-element
    pass_type = None
    cross_type = None
    for f in play_type.fields:
        if f.name == "Pass" and isinstance(f.dataType, StructType):
            pass_type = f.dataType
        elif f.name == "Cross" and isinstance(f.dataType, StructType):
            cross_type = f.dataType
    
    pass_fields = set(f.name for f in pass_type.fields) if pass_type else set()
    cross_fields = set(f.name for f in cross_type.fields) if cross_type else set()
    
    for attr, alias, dtype in pass_attrs:
        if attr in pass_fields:
            select_exprs.append(col(f"{play_path}.Pass.{attr}").alias(alias))
        else:
            select_exprs.append(lit(None).cast(dtype).alias(alias))
    
    for attr, alias, dtype in cross_attrs:
        if attr in cross_fields:
            select_exprs.append(col(f"{play_path}.Cross.{attr}").alias(alias))
        else:
            select_exprs.append(lit(None).cast(dtype).alias(alias))
    
    return event_df.filter(col(play_path).isNotNull()).select(*select_exprs)

# Build a DataFrame for each nesting level
play_dfs = []
for path, context in play_levels:
    df = build_play_df(event_df, path, context)
    if df is not None:
        play_dfs.append(df)
        print(f"  {context}: {path} -> {df.count()} rows")

# Union all play DataFrames
plays_flat_df = play_dfs[0]
for df in play_dfs[1:]:
    plays_flat_df = plays_flat_df.unionByName(df, allowMissingColumns=True)

print(f"\nTotal plays: {plays_flat_df.count()}")
print("\nPlays by context:")
plays_flat_df.groupBy("play_context").count().orderBy(col("count").desc()).show()

In [0]:
# ── 3. Save to Delta tables ──
# Using overwrite + overwriteSchema to replace existing data and schema.
# For future incremental file additions, switch mode to "append".

catalog = "`bundesliga-2022-2023`"
schema_name = f"{catalog}.batch"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

# 1. Events flat table
(
    events_flat_df.write
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.raw_events")
)

# 2. Plays flat table
(
    plays_flat_df.write
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.raw_plays")
)

print("Tables created:")
for t in ["raw_events", "raw_plays"]:
    count = spark.table(f"{schema_name}.{t}").count()
    print(f"  {schema_name}.{t}: {count} rows")

In [0]:
# ── 4. Validate: all XML fields are now captured in Delta tables ──

from pyspark.sql.types import StructType, ArrayType

# 4a. Check raw_plays for possession data
print("=== raw_plays: possession-related columns ===")
plays_tbl = spark.table("`bundesliga-2022-2023`.batch.raw_plays")
poss_play_cols = [c for c in plays_tbl.columns if "poss" in c.lower()]
print(f"  Found: {poss_play_cols}")
print(f"  Non-null ball_possession_phase: {plays_tbl.filter(col('ball_possession_phase').isNotNull()).count()}")

# 4b. Check raw_events for possession data
print("\n=== raw_events: possession-related columns ===")
events_tbl = spark.table("`bundesliga-2022-2023`.batch.raw_events")
poss_ev_cols = [c for c in events_tbl.columns if "poss" in c.lower()]
print(f"  Found: {poss_ev_cols}")
for pc in poss_ev_cols:
    non_null = events_tbl.filter(col(pc).isNotNull()).count()
    print(f"  Non-null {pc}: {non_null}")

# 4c. Verify ALL possession fields from raw XML are now captured
print("\n=== Possession fields: raw XML vs Delta tables ===")
recovered = 0
still_lost = 0
for field in event_df.schema.fields:
    if field.name.startswith("_"):
        continue
    if isinstance(field.dataType, StructType):
        for sub_field in field.dataType.fields:
            if "poss" in sub_field.name.lower():
                # Check if a matching column exists in events_tbl or plays_tbl
                prefix = to_snake(field.name)
                col_name = to_snake(sub_field.name)
                full_name = f"{prefix}__{col_name}"
                if full_name in events_tbl.columns or col_name in plays_tbl.columns:
                    print(f"  {field.name}.{sub_field.name} -> {full_name} ✅ CAPTURED")
                    recovered += 1
                else:
                    print(f"  {field.name}.{sub_field.name} -> ⚠️ STILL LOST")
                    still_lost += 1

print(f"\n  Possession fields recovered: {recovered}")
print(f"  Possession fields still lost: {still_lost}")

# 4d. Full audit: every non-attribute child field in event_df vs Delta tables
print("\n=== Full audit: all child fields in raw XML vs Delta tables ===")
event_cols_set = set(events_tbl.columns)
plays_cols_set = set(plays_tbl.columns)

total_child_fields = 0
captured_fields = 0
lost_fields = []
for field in event_df.schema.fields:
    if field.name.startswith("_"):
        continue
    if isinstance(field.dataType, StructType):
        for sub_field in field.dataType.fields:
            if isinstance(sub_field.dataType, StructType):
                # Nested struct - check its children
                for nested_field in sub_field.dataType.fields:
                    if isinstance(nested_field.dataType, StructType):
                        for deep_field in nested_field.dataType.fields:
                            total_child_fields += 1
                            prefix = to_snake(field.name)
                            col_name = to_snake(sub_field.name)
                            nested_col = to_snake(nested_field.name)
                            deep_col = to_snake(deep_field.name)
                            full_name = f"{prefix}__{col_name}__{nested_col}__{deep_col}"
                            if full_name in event_cols_set:
                                captured_fields += 1
                            else:
                                lost_fields.append(full_name)
                    else:
                        total_child_fields += 1
                        prefix = to_snake(field.name)
                        col_name = to_snake(sub_field.name)
                        nested_col = to_snake(nested_field.name)
                        full_name = f"{prefix}__{col_name}__{nested_col}"
                        if full_name in event_cols_set:
                            captured_fields += 1
                        else:
                            lost_fields.append(full_name)
            else:
                total_child_fields += 1
                prefix = to_snake(field.name)
                col_name = to_snake(sub_field.name)
                full_name = f"{prefix}__{col_name}"
                if full_name in event_cols_set or (field.name == "Play" and col_name in plays_cols_set):
                    captured_fields += 1
                else:
                    lost_fields.append(full_name)

print(f"  Total child fields in raw XML: {total_child_fields}")
print(f"  Captured in Delta tables: {captured_fields}")
print(f"  Still lost: {len(lost_fields)}")
if lost_fields:
    print(f"\n  ⚠️ Lost fields: {lost_fields}")
else:
    print(f"\n  ✅ ZERO information loss — all fields captured!")

print(f"\n  raw_events columns: {len(events_tbl.columns)}")
print(f"  raw_plays columns: {len(plays_tbl.columns)}")
print(f"  raw_events rows: {events_tbl.count()}")
print(f"  raw_plays rows: {plays_tbl.count()}")

In [0]:
# ── 5. Example: play ball possession phase data ──
# ball_possession_phase is a sequential integer identifying which possession phase
# a play belongs to. Each time possession changes between teams, the phase number increments.
# Group by match_id + ball_possession_phase to get per-phase stats.

from pyspark.sql.functions import col

plays = spark.table("`bundesliga-2022-2023`.batch.raw_plays")

display(
    plays.select(
        "match_id", "event_id", "event_time", "play_context",
        "player", "team", "ball_possession_phase", "evaluation",
        "play_origin", "height", "distance"
    )
    .filter(col("ball_possession_phase").isNotNull())
    .orderBy("match_id", "ball_possession_phase")
    .limit(25)
)

In [0]:
# ── 7. Example: pass data and shot data ──

from pyspark.sql.functions import col

# ── Passes: plays where pass_direction is not null ──
print("=== Sample passes (raw_plays, pass_direction IS NOT NULL) ===")
plays = spark.table("`bundesliga-2022-2023`.batch.raw_plays")
display(
    plays.select(
        "match_id", "event_id", "event_time", "play_context",
        "player", "recipient", "team", "evaluation",
        "x_source_position", "y_source_position",
        "x_position", "y_position",
        "play_origin", "height", "distance",
        "pass_direction", "pass_one_two", "pass_free_kick_layup"
    )
    .filter(col("pass_direction").isNotNull())
    .orderBy("match_id", "event_time")
    .limit(25)
)

# ── Shots: ShotAtGoal events from raw_events ──
# ShotAtGoal has ~50 fields – showing ALL of them
print("\n=== Sample shots at goal (raw_events, event_type = ShotAtGoal) ===")
events = spark.table("`bundesliga-2022-2023`.batch.raw_events")
display(
    events.filter(col("event_type") == "ShotAtGoal")
    .select(
        # Event attributes
        "match_id", "event_id", "event_time",
        "x_position", "y_position",
        "x_source_position", "y_source_position",
        # Shot identity & context
        "shot_at_goal__player",
        "shot_at_goal__team",
        "shot_at_goal__ball_possession_phase",
        "shot_at_goal__type_of_shot",
        "shot_at_goal__extended_type_of_shot",
        "shot_at_goal__other_shot",
        "shot_at_goal__shot_origin",
        "shot_at_goal__setup_origin",
        "shot_at_goal__build_up",
        "shot_at_goal__after_free_kick",
        # Shot quality metrics
        "shot_at_goal__x_g",
        "shot_at_goal__distance_to_goal",
        "shot_at_goal__angle_to_goal",
        "shot_at_goal__goal_distance_goalkeeper",
        "shot_at_goal__inside_box",
        "shot_at_goal__amount_of_defenders",
        "shot_at_goal__pressure",
        "shot_at_goal__player_speed",
        # Shot evaluation
        "shot_at_goal__chance_evaluation",
        "shot_at_goal__shot_condition",
        "shot_at_goal__shot_contribution",
        "shot_at_goal__significance_evaluation",
        "shot_at_goal__sitter_contribution",
        "shot_at_goal__counter_attack",
        # Taker info
        "shot_at_goal__taker_setup",
        "shot_at_goal__taker_ball_control",
        # Assist info (153 non-null – was missing!)
        "shot_at_goal__assist_action",
        "shot_at_goal__assist_shot_at_goal",
        "shot_at_goal__assist_type_shot_at_goal",
        # Goal outcome
        "shot_at_goal__successful_shot__current_result",
        "shot_at_goal__successful_shot__goal_zone",
        "shot_at_goal__successful_shot__assist",
        "shot_at_goal__successful_shot__assist_type",
        "shot_at_goal__successful_shot__assist_contribution",
        "shot_at_goal__successful_shot__solo",
        "shot_at_goal__successful_shot__deflection_keeper",
        "shot_at_goal__successful_shot__deflection_player",
        "shot_at_goal__successful_shot__error",
        "shot_at_goal__successful_shot__ref_decision_evaluation",
        # Save outcome
        "shot_at_goal__saved_shot__goal_keeper",
        "shot_at_goal__saved_shot__save_type",
        "shot_at_goal__saved_shot__save_result",
        "shot_at_goal__saved_shot__save_evaluation",
        # Miss outcome
        "shot_at_goal__shot_wide__placing",
        "shot_at_goal__shot_wide__pitch_marking",
        "shot_at_goal__shot_wood_work__location",
        # Block outcome
        "shot_at_goal__blocked_shot__player",
        "shot_at_goal__blocked_shot__blocked_by_own_team",
        "shot_at_goal__blocked_shot__goal_prevented",
    )
    .orderBy("match_id", "event_time")
    .limit(25)
)

In [0]:
# ── 6. Build possession sequences by detecting team changes ──
# ball_possession_phase is a per-event ID (1:1), not a grouping key.
# Real possession sequences are built by grouping consecutive plays
# by the same team until possession changes.

from pyspark.sql.functions import col, when, lit, sum as spark_sum, count as spark_count, lag, concat, min, max
from pyspark.sql.window import Window

plays = spark.table("`bundesliga-2022-2023`.batch.raw_plays")

# Order plays within each match by event_time
w = Window.partitionBy("match_id").orderBy("event_time", "event_id")

# Detect team change: 1 when team differs from previous play, else 0
plays_with_change = plays.filter(col("team").isNotNull()) \
    .withColumn("prev_team", lag(col("team"), 1).over(w)) \
    .withColumn("is_change", when(col("team") != col("prev_team"), lit(1)).otherwise(lit(0)))

# Cumulative sum of changes = possession sequence ID within each match
w_cum = Window.partitionBy("match_id").orderBy("event_time", "event_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)

plays_with_seq = plays_with_change \
    .withColumn("possession_seq", spark_sum("is_change").over(w_cum)) \
    .withColumn("possession_id", concat(col("match_id"), lit("_P"), col("possession_seq")))

# Show sample: first 25 plays with their possession sequence
print("=== Sample: plays with possession sequence (match DFL-MAT-J03WMX) ===")
display(
    plays_with_seq
    .select(
        "match_id", "event_id", "event_time", "play_context",
        "player", "team", "possession_seq", "evaluation",
        "play_origin", "height", "distance"
    )
    .filter(col("match_id") == "DFL-MAT-J03WMX")
    .orderBy("event_time")
    .limit(25)
)

# Aggregate: one row per possession sequence
print("\n=== Possession sequences summary (top 15 by play count) ===")
possession_summary = plays_with_seq \
    .groupBy("match_id", "possession_seq", "team") \
    .agg(
        spark_count("*").alias("n_plays"),
        spark_count(when(col("evaluation") == "successfullyCompleted", True)).alias("successful"),
        spark_count(when(col("evaluation") == "unsuccessful", True)).alias("unsuccessful"),
        min("event_time").alias("seq_start"),
        max("event_time").alias("seq_end"),
    ) \
    .orderBy(col("n_plays").desc())

display(possession_summary.select("match_id", "possession_seq", "team", "n_plays", "successful", "unsuccessful", "seq_start", "seq_end").limit(15))

print("\n=== Distribution of plays per possession sequence ===")
possession_summary.groupBy("n_plays").count().orderBy("n_plays").show()

# Save as a new Delta table
print("\n=== Saving possession_sequences table ===")
plays_with_seq.write \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("`bundesliga-2022-2023`.batch.possession_sequences")

n_seqs = spark.table("`bundesliga-2022-2023`.batch.possession_sequences") \
    .select("match_id", "possession_seq").distinct().count()
print(f"  Saved: {n_seqs} unique possession sequences across all matches")
print(f"  Table: `bundesliga-2022-2023`.batch.possession_sequences")